### RAG Pipleine- Data Ingestion to Vector DB pipeline

In [6]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [8]:
### Read all the pdf's inside the directory

def process_all_pdfs(pdf_directory):
    """ Process all PDF files in a directory """
    
    all_documents= []
    pdf_dir= Path(pdf_directory)

    ## Find all pdf files recursively
    pdf_files= list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing:{pdf_file.name}")
        try:
            loader= PyPDFLoader(str(pdf_file))
            documents= loader.load()

            # Add source information to metadata
            for doc in documents: 
                doc.metadata['source_file']= pdf_file.name
                doc.metadata['file_type']= 'pdf'

            all_documents.extend(documents)
            print(f" Loaded{len(documents)} pages")
        
        except Exception as e:
            print(f" Error:{e}")
    
    print(f"\n Total documents loaded: {len(all_documents)}")
    return all_documents

# Process all Pdf's in the data directory
all_pdf_documents= process_all_pdfs("../data")

Found 2 PDF files to process

Processing:1706.03762v7.pdf
 Loaded15 pages

Processing:1-s2.0-S1877050925029461-main.pdf
 Loaded10 pages

 Total documents loaded: 25


In [9]:
all_pdf_documents

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '../data/pdf/1706.03762v7.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': '1706.03762v7.pdf', 'file_type': 'pdf'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of

In [14]:
### Text Splitting get in to chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """ Split documents into smaller chunks for better RAG Performance   """

    text_splitter= RecursiveCharacterTextSplitter(
        chunk_size= chunk_size, 
        chunk_overlap= chunk_overlap,
        length_function= len, 
        separators= ["\n\n", "\n"," ",""]
    )

    split_docs = text_splitter.split_documents(documents)
    print( f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk

    if split_docs: 
        print(f"\m Example chunk:")
        print(f"Content:{split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

<>:19: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<>:19: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
/var/folders/0f/k8vbmph94wl3bj51kr2v5p3h0000gn/T/ipykernel_84094/3952028602.py:19: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
  print(f"\m Example chunk:")


In [16]:
chunks = split_documents(all_pdf_documents)
chunks

Split 25 documents into 117 chunks
\m Example chunk:
Content:Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
...
Metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '../data/pdf/1706.03762v7.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': '1706.03762v7.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '../data/pdf/1706.03762v7.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': '1706.03762v7.pdf', 'file_type': 'pdf'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of

## Embedding And Vector StoreDB 

In [31]:
# %pip install chromadb
# %pip install sentence_transformers
import numpy as np 
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [36]:
class EmbeddingManager:
    """ Handles document embedding generation using SentenceTransformer """

    def __init__(self, model_name:str= "all-MiniLM-L6-v2"):
        """ 
        Initialize the embedding manager

        Args: 
            model_name: HuggingFace model name for sentenct embeddings
        """

        self.model_name= model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """ Load the SentenceTransformer model """

        try: 
            print(f" Loading embedding model: {self.model_name}")
            self.model= SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension :{self.model.get_sentence_embedding_dimension()}")
        except Exception as e: 
            print(f"Error loading model {self.model_name} :{e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """ 
        Generate embeddings for a list of texts

        Args: 
            texts: List of text strings to embed

        Returns: 
            numpy array of embeddings with shape (len(texts), embedding_dim)

        """

        if not self.model:
            raise ValueError("Model not loaded")

        
        print(f" Generating embeddings for {len(texts)} texts....")
        embeddings= self.model.encode(texts, show_progress_bar= True)
        print(f"Generated embeddings with shape: {embeddings.shape}")

        return embeddings
'''
    def get_embedding_dimension(self) -> int: 
        """ Get the embedding dimension of the model """ 

        if not self.model: 
            raise ValueError("Model not loaded")
        return self.model.get_sentence_embedding_dimension()
'''
## initalize the embedding manager

embedding_manager= EmbeddingManager()
embedding_manager

 Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9360.26it/s]


Model loaded successfully. Embedding dimension :384


/var/folders/0f/k8vbmph94wl3bj51kr2v5p3h0000gn/T/ipykernel_84094/4003120065.py:22: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension :{self.model.get_sentence_embedding_dimension()}")


## VectorStore

In [70]:
class VectorStore:
    """ Manages document embedding in a ChromaDB  vector store. """

    def __init__(self, collection_name: str= "pdf_documents", persist_directory:str ="../data/vector_store"):
        """
        Initialize the vector store

        Args: 
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to presist the vector store
        """

        self.collection_name= collection_name
        self.persist_directory = persist_directory
        self.client= None
        self.collection= None
        self._intialize_store()

    
    def _intialize_store(self):
        """ Initialize ChromDB client and collection """

        try: 
            #create persistent ChromaDB client
            os.makedirs(self.persist_directory,exist_ok= True)
            self.client = chromadb.PersistentClient(path =self.persist_directory)

            # Get or create collection
            self.collection =self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )

            print(f"Vector store initiliazed. Collection:{self.collection_name}")
            print(f"Existing Documents in collection :{self.collection.count()}")

        except Exception as e:
            print(f"Error initalizing vector store:{e}")
            raise
    
    def add_documents(self,documents: List[Any],embeddings:np.array):
        """ 
        Add documents and their embeddings to the vector store

        Args: 
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        
        """

        if len(documents)!= len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store")

        ## Prepare data for ChromaDB

        ids=[] 
        metadatas= []
        documents_text=[]
        embeddings_list=[]

        for i, (doc,embedding) in enumerate(zip(documents, embeddings)):
            ## Generate unique id
            doc_id= f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata= dict(doc.metadata)
            metadata['doc_index']=i 
            metadata['content_length']= len(doc.page_content)
            metadatas.append(metadata)

            #Document Content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

            ##Add to Collection
            try:
                self.collection.add(
                    ids= ids, 
                    embeddings= embeddings_list,
                    metadatas= metadatas,
                    documents= documents_text)
                
                print(f"Successfully added {len(documents)} documents to vector store")
                print(f"Total documents in collection: {self.collection.count()}")

            except Exception as e:
                print(f"Error adding documents to vectore store : {e}")
                raise


vectorstore= VectorStore()
vectorstore

Vector store initiliazed. Collection:pdf_documents
Existing Documents in collection :1


In [48]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '../data/pdf/1706.03762v7.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': '1706.03762v7.pdf', 'file_type': 'pdf'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of

In [49]:
### Convert the text to embeddings 

texts= [doc.page_content for doc in chunks]
texts

['Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture, the Transformer,\nbased solely on attention mechanisms, dispensing with recurrenc

In [71]:
### Convert the text to embeddings 

texts= [doc.page_content for doc in chunks]

## Generate the embeddings

embeddings= embedding_manager.generate_embeddings(texts)

## Store in the vector database

vectorstore.add_documents(chunks,embeddings)

 Generating embeddings for 117 texts....


Batches: 100%|██████████| 4/4 [00:00<00:00,  8.72it/s]


Generated embeddings with shape: (117, 384)
Adding 117 documents to vector store
Successfully added 117 documents to vector store
Total documents in collection: 2
Successfully added 117 documents to vector store
Total documents in collection: 3
Successfully added 117 documents to vector store
Total documents in collection: 4
Successfully added 117 documents to vector store
Total documents in collection: 5
Successfully added 117 documents to vector store
Total documents in collection: 6
Successfully added 117 documents to vector store
Total documents in collection: 7
Successfully added 117 documents to vector store
Total documents in collection: 8
Successfully added 117 documents to vector store
Total documents in collection: 9
Successfully added 117 documents to vector store
Total documents in collection: 10
Successfully added 117 documents to vector store
Total documents in collection: 11
Successfully added 117 documents to vector store
Total documents in collection: 12
Successfully a

## Retriever Pipeline From VectorStore

In [82]:
class RAGRetriever:
    """ Handles query-based retrieval from the vector store  """

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """ 
        Initialize the retriever
        
        Args: 
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query:str , top_k: int=5 , score_threshold:float= 0.0) -> List[Dict[str,Any]]:
        """ 
        Retrieve relevant documents for a query 

        Args: 
            query: the search query
            top_k: Number of top results to return 
            score_threshold: Minimum similarity score threshold

        Returns: 
            List of dictionaries containing documents and metadta
        """

        # Generate query embedding
        query_embedding= self. embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try: 
            results= self.vector_store.collection.query(
                query_embeddings= [query_embedding.tolist()],
                n_results= top_k)

            #process resuls
            retrieved_docs= []
        
            if results['documents'] and results['documents'][0]:
                documents= results['documents'][0]
                metadatas= results['metadatas'][0]
                distances = results['distances'][0]
                ids= results['ids'][0]

                for i,(doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):

                    #convert distance to Similarity score(ChromoDB uses cosine distance)
                    similarity_score= 1-distance
                    
                    if similarity_score>= score_threshold:
                        retrieved_docs.append({
                            'id':doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score, 
                            'distance': distance,
                            'rank': i+1
                        })
                print(f" Reterived {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval:{e}")
            return []

rag_retriever= RAGRetriever(vectorstore, embedding_manager)

In [83]:
rag_retriever

In [85]:
rag_retriever.retrieve("What is Lung Cancer")

 Generating embeddings for 1 texts....


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.26it/s]

Generated embeddings with shape: (1, 384)
 Reterived 3 documents (after filtering)


[{'id': 'doc_9eac7eac_62',
  'content': '1518 Christoph Mattmann  et al. / Procedia Computer Science 270 (2025) 1517–1526\n2 Christoph Mattmann et al. / Procedia Computer Science 00 (2025) 000–000\n1. Introduction\nLung cancer is one of the most common types of cancer and also one of the deadliest [9, 45, 42]. The later lung\ncancer is detected, the higher the mortality rate [45, 17]. Early signs of lung cancer are difficult to detect as there\nare no clear symptoms [32]. Only small nodules in the lung, often between 10 mm and 30 mm in diameter, can be\nmalignant and indicate lung cancer [32, 20, 52]. To identify these nodules, a CT scan is one of the most common\nperformed methods [43]. Detecting these nodules is a challenging and time-consuming task to check all layers to find\nnodules as their size, appearance and location vary [32, 31, 36]. To avoid missing malignant nodules, in particular,\nCAD can be integrated into the diagnostic process [46, 13, 20]. The development of CAD syst

In [ ]:
rag_retriever.retrieve("What is attention all you need")

 Generating embeddings for 1 texts....


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.13it/s]

Generated embeddings with shape: (1, 384)
 Reterived 1 documents (after filtering)


[{'id': 'doc_3a7f43ce_12',
  'content': '3.2 Attention\nAn attention function can be described as mapping a query and a set of key-value pairs to an output,\nwhere the query, keys, values, and output are all vectors. The output is computed as a weighted sum\n3',
  'metadata': {'moddate': '2024-04-10T21:11:43+00:00',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'trapped': '/False',
   'doc_index': 12,
   'creationdate': '2024-04-10T21:11:43+00:00',
   'producer': 'pdfTeX-1.40.25',
   'page': 2,
   'file_type': 'pdf',
   'creator': 'LaTeX with hyperref',
   'subject': '',
   'total_pages': 15,
   'source_file': '1706.03762v7.pdf',
   'content_length': 216,
   'keywords': '',
   'author': '',
   'title': '',
   'source': '../data/pdf/1706.03762v7.pdf',
   'page_label': '3'},
  'similarity_score': 0.2098538875579834,
  'distance': 0.7901461124420166,
  'rank': 1}]

## Integration Vectordb Context pipeline with LLM output

In [102]:
## Simple RAG Pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

## Initialize the Groq LLM (set your GROQ_API_KEY in enviornment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm =ChatGroq(groq_api_key=groq_api_key,model_name='openai/gpt-oss-20b', temperature= 0.1, max_tokens=1024)


## 2. Simple RAG function: retrieve context + generate response 

def rag_simple(query, retriever, llm, top_k=3):
    ## retriever the context
    results= retriever.retrieve(query,top_k= top_k)
    context= "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found"

    ## generate the answer using GROQ LLM

    prompt= f""" Use the following context to answer the question concisely 

                Context:
                {context}

                Question: {query}

                Answer: """
    response = llm.invoke([prompt.format(context=context, query= query)])
    return response.content

In [103]:
answer= rag_simple("What is attention mechanism?", rag_retriever,llm)
print(answer)

 Generating embeddings for 1 texts....


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.08it/s]


Generated embeddings with shape: (1, 384)
 Reterived 2 documents (after filtering)
The attention mechanism is a function that takes a **query** vector and a set of **key‑value** pairs and produces an **output** vector. It does this by computing a weighted sum of the values, where the weights are derived from the similarity (or compatibility) between the query and each key. This allows the model to focus on relevant parts of the input when generating each output.


## Enhanced RAG Pipeline Features

In [ ]:
# ------ Enhanced RAG pipleine Features ----- 

def rag_advanced(query, retriever,llm, top_k=5, min_score=0.2, return_context=False): 
    """ 
    RAG pipeline with extra features: 
    - Returns answer, sources, confidence scorem and optinally full context
    """

    results= retriever.retrieve(query, top_k= top_k,  score_threshold= min_score)
    if not results:
            return {'answer': 'No relevant context found.', 'sources':[], 'confidence':0.0 ,'context':''}
    
    #Prepare context and sources

    context= "\n\n".join([doc['content'] for doc in results])
    sources= [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source','unknown')), 
        'page': doc['metadata'].get('page','unknown'), 
        'score': doc['similarity_score'], 
        'preview':doc['content'][:300]+ '...'
    } for doc in results ]

    confidence= max([doc['similarity_score'] for doc in results])

    # Generate answer

    prompt= f""" Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion:{query}\n\nAnswer:"""
    response= llm.invoke([prompt.format(context=context,query=query)])
    
    output= {
            'answer': response.content,
            'sources': sources, 
            'confidence': confidence
            }

    
    if return_context: 
        output['context']= context
    return output

#Example usage
result= rag_advanced("What is lung cancer? ", rag_retriever,llm, top_k=3, min_score= 0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

 Generating embeddings for 1 texts....


Batches: 100%|██████████| 1/1 [00:00<00:00, 16.42it/s]

Generated embeddings with shape: (1, 384)
 Reterived 1 documents (after filtering)


Answer: Lung cancer is a malignant tumor that develops in the tissues of the lungs. It is one of the most common and deadliest cancers, with early signs hard to detect because malignant nodules are often small (10–30 mm) and asymptomatic. Early detection is critical, as later-stage disease carries a much higher mortality rate.
Sources: [{'source': '1-s2.0-S1877050925029461-main.pdf', 'page': 1, 'score': 0.1511935591697693, 'preview': '1518 Christoph Mattmann  et al. / Procedia Computer Science 270 (2025) 1517–1526\n2 Christoph Mattmann et al. / Procedia Computer Science 00 (2025) 000–000\n1. Introduction\nLung cancer is one of the most common types of cancer and also one of the deadliest [9, 45, 42]. The later lung\ncancer is detecte...'}]
Confidence: 0.1511935591697693
Context Preview: 1518 Christoph Mattmann  et al. / Procedia Computer Science 270 (2025) 1517–1526
2 Christoph Mattmann et al. / Procedia Computer Science 00 (2025) 000–000
1. Introduction
Lung cancer is one of the most 

In [115]:
result= rag_advanced("What is attention mechanism? ", rag_retriever,llm, top_k=3, min_score= 0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

 Generating embeddings for 1 texts....


Batches: 100%|██████████| 1/1 [00:00<00:00,  9.73it/s]

Generated embeddings with shape: (1, 384)
 Reterived 2 documents (after filtering)


Answer: The attention mechanism is a neural network component that, given a query vector and a set of key‑value pairs, produces an output as a weighted sum of the values. The weights are computed from the similarity between the query and each key, allowing the model to focus on relevant parts of the input (e.g., long‑distance dependencies in a sentence).
Sources: [{'source': '1706.03762v7.pdf', 'page': 2, 'score': 0.2714170217514038, 'preview': '3.2 Attention\nAn attention function can be described as mapping a query and a set of key-value pairs to an output,\nwhere the query, keys, values, and output are all vectors. The output is computed as a weighted sum\n3...'}, {'source': '1706.03762v7.pdf', 'page': 12, 'score': 0.13597571849822998, 'preview': 'Attention Visualizations\nInput-Input Layer5\nIt\nis\nin\nthis\nspirit\nthat\na\nmajority\nof\nAmerican\ngovernments\nhave\npassed\nnew\nlaws\nsince\n2009\nmaking\nthe\nregistration\nor\nvoting\nprocess\nmore\ndifficult\n.\n<EOS>\n<pad>\n<

In [116]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is attention is all you need", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

 Generating embeddings for 1 texts....


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.83it/s]

Generated embeddings with shape: (1, 384)
 Reterived 1 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
3.2 Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output,
where the query, keys, values, and output are all vectors. The output is computed as a weighted sum
3

Question: what is attention is al

l you need

Answer:

Final Answer: **Attention Is All You Need** is a landmark 2017 paper that introduced the Transformer architecture. It shows that a model built entirely from self‑attention mechanisms—without any recurrent or convolutional layers—can learn long‑range dependencies and achieve state‑of‑the‑art performance on tasks like machine translation. The key idea is to use multi‑head self‑attention to compute weighted sums of key‑value pairs for each query, enabling parallel, efficient sequence modeling.

Citations:
[1] 1706.03762v7.pdf (page 2)
Summary: The 2017 paper “Attention Is All You Need” introduced the Transformer, a model that relies solely on self‑attention mechanisms—eschewing recurrence and convolution—to capture long‑range dependencies. By using multi‑head self‑attention to compute weighted sums of key‑value pairs for each query, the architecture enables parallel, efficient sequence modeling and achieves state‑of‑the‑art results on tasks such as machine translation